### **Extraer el contenido del sitioweb y exportarlo a un archivo `HTML`**

Vamos a trabajar en el enlace:

- https://quotes.toscrape.com/page/1/
- https://quotes.toscrape.com/page/2/

#### **Crear la spider**

Dado que en el archivo **`1-quotes_to_scrape.ipynb`** realizamos todo el proceso de creación desde el Entorno virtual, instalación de Scrapy y creación del proyecto, ahora solo nos queda comenzar a crear nuestras spiders. Para ello desde la terminal ejecutamos lo siguiente:

<center><img src="https://i.postimg.cc/25nKGzHK/ws-377.png"></center>

Luego, debemos editar el archivo `quotes_to_scrape_dos.py` que se crea, con el código que se muestra abajo.

#### **Código**

Las Spiders son clases que definimos y que Scrapy utiliza para extraer información de un sitio web (o de un grupo de sitios web). Deben llamar a **`Spider`** como una subclase y definir las requests iniciales a realizar, opcionalmente cómo seguir (follow) enlaces en las páginas, y cómo parsear (parse) el contenido de la página descargada para extraer datos.

In [ ]:
from pathlib import Path

import scrapy


class QuotesToScrapeDosSpider(scrapy.Spider):
    name = "quotes_to_scrape_dos"

    def start_requests(self):
        urls = [
            "https://quotes.toscrape.com/page/1/",
            "https://quotes.toscrape.com/page/2/",
        ]
        for url in urls:
            yield scrapy.Request(url=url, callback=self.parse)

    def parse(self, response):
        page = response.url.split("/")[-2]
        filename = f"quotes-{page}.html"
        Path(filename).write_bytes(response.body)
        self.log(f"Saved file {filename}")

Como puedes ver, nuestro Spider subclasea **`scrapy.Spider`** y define algunos atributos y métodos:

- **`name`**: identifica al Spider. Debe ser único dentro de un proyecto, es decir, no se puede establecer el mismo nombre para diferentes Spiders.

- **`start_requests()`**: debe devolver un iterable de Requests (puedes devolver una lista de requests o escribir una generator function) a partir del cual la Spider comenzará a crawlear (rastrear). Las requests subsiguientes se generarán sucesivamente a partir de estas requests iniciales.

- **`parse()`**: un método que será llamado para manejar la response descargada para cada una de las requests realizadas. El parámetro response es una instancia de **`TextResponse`** que contiene el contenido de la página y dispone de otros métodos útiles para manejarlo.

El método **`parse()`** normalmente parsea la response, extrayendo los datos scrapeados como dicts y también encontrando nuevas URLs a seguir y creando nuevas requests (**`Request`**) a partir de ellas.

#### **Ejecución**

Nos posicionamos en la ruta correcta y ejecutamos nuestra Spider:

In [ ]:
scrapy crawl quotes_to_scrape_dos

<center><img src="https://i.postimg.cc/d0S6d6Tn/ws-378.png"></center>

También obtenemos los mensajes enviados por el método `.log()`:

<center><img src="https://i.postimg.cc/vHdzWccR/ws-379.png"></center>

Ahora, comprobemos los archivos en el directorio actual. Observaremos que se han creado dos nuevos archivos: `quotes-1.html` y `quotes-2.html`, con el contenido de las respectivas URL, tal y como indica nuestro método `parse`.

<center><img src="https://i.postimg.cc/t4xQTBvd/ws-380.png"></center>

#### **¿Qué ha pasado tras bambalinas?**

Scrapy programa los objetos `scrapy.Request` devueltos por el método `start_requests` del Spider. Al recibir una respuesta (response) para cada una de ellas, instancia objetos `Response` y llama al método `callback` asociado a la request (en este caso, el método `parse`) pasando la response como argumento.

#### **Un atajo al método start_requests (Modificación de nuestro código)**

En lugar de implementar un método `start_requests()` que genere objetos scrapy.Request a partir de URLs, puedes simplemente definir un atributo de clase `start_urls` con una lista de URLs. Esta lista será utilizada por la implementación por defecto de `start_requests()` para crear las requests iniciales para tu spider.

In [ ]:
from pathlib import Path

import scrapy


class QuotesToScrapeTresSpider(scrapy.Spider):
    name = "quotes_to_scrape_tres"
    start_urls = [
        "https://quotes.toscrape.com/page/1/",
        "https://quotes.toscrape.com/page/2/",
    ]

    def parse(self, response):
        page = response.url.split("/")[-2]
        # Le sumamos '2' para que se creen 2 nuevos archivos y no sobreescriban los anteriores
        filename = f"quotes-{int(page)+2}.html"
        Path(filename).write_bytes(response.body)

El método `parse()` será llamado para manejar cada una de las requests para esas URLs, aunque no le hayamos dicho explícitamente a Scrapy que lo haga. Esto sucede porque `parse()` es el método `callback` por defecto de Scrapy, que es llamado para las requests sin un callback explícitamente asignado.

#### **Crear una nueva spider y ejecutarla**

Desde la terminal ejecutamos lo siguiente:

<center><img src="https://i.postimg.cc/fTvMs2FV/ws-381.png"></center>

Luego, debemos editar el archivo `quotes_to_scrape_tres.py` que se crea, con el nuevo código. Luego, desde la terminal nos posicionamos en la ruta correcta y ejecutamos nuestra Spider:

In [ ]:
scrapy crawl quotes_to_scrape_tres

<center><img src="https://i.postimg.cc/G3v4819t/ws-382.png"></center>

Ahora, comprobemos los archivos en el directorio actual. Observaremos que se han creado dos nuevos archivos: `quotes-3.html` y `quotes-4.html`, con el contenido de las respectivas URL, tal y como indica nuestro método `parse`.

<center><img src="https://i.postimg.cc/4Ndn8HB7/ws-383.png"></center>